# TinyStories

In [ ]:
from torch import nn

from artifacts.core.artifact import Resources
from artifacts.core.artifact import resolve
from training.lr_schedule import WarmupCosine
from training.optimizers import AdamW
from training.leg import LoopConfig, TrainingParameters
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.stages.pretraining.evaluations import TokenLoss
from artifacts.stages.pretraining.samplers import RandomWindows
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import Transformer
from artifacts.profiles import Profile
from profiling.profile_events import launch_chain
from profiling.util import run_profile

In [2]:
HF = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
train = SourceURL(name="tinystories-train", url=f"{HF}/TinyStoriesV2-GPT4-train.txt")
valid = SourceURL(name="tinystories-valid", url=f"{HF}/TinyStoriesV2-GPT4-valid.txt")

tokenizer = BPETokenizer(
    vocab_size=10000,
    special_tokens=("<|endoftext|>",),
    sources=(valid,),
    allocated_resources=Resources(cpu=4),
)

dataset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(train,),
    valid_sources=(valid,),
)

pretraining = Pretraining(
    run_id="runofex",
    dataset=dataset,
    sampler=RandomWindows(),
    autocast=None,  # the T4 has no bf16 unit; "torch.bfloat16" on an L4, A10 or newer
    model=Transformer(
        vocab_size=tokenizer.vocab_size,
        d_model=64,
        d_ff=128,
        sequence_length=32,
        num_layers=4,
        num_heads=16,
        rope_theta=10000,
        tie_embeddings=False,
        dtype="torch.float32",
    ),
    training_parameters=TrainingParameters(
        total_steps=20000,
        batch_size=64,
        max_norm=1.0,
        lr_schedule=WarmupCosine(
            max_learning_rate=3e-4,
            min_learning_rate=3e-5,
            warmup_iters=200,
            cosine_cycle_iters=5000,
        ),
        optimizer=AdamW(
            betas=(0.9, 0.95),
            weight_decay=0.1,
            eps=1e-8,
        ),
        seed=0,
    ),
    loop_config=LoopConfig(
        checkpoint_every=5000,
        evaluations=(TokenLoss(eval_every=200, batch_size=64),),
        gpu_check_every=200,
        compile=None,
    ),
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
    tags={"experiment": ["myfirstexperiment"]},
)

## Declare

In [3]:
for artifact in resolve(pretraining):
    print(f"{type(artifact).__name__:16} {artifact.artifact_path}")

SourceURL        sources/tinystories-train
SourceURL        sources/tinystories-valid
Tokenizer        tokenizers/bpe-10000-b46983c6f3
TokenizedSource  tokenized/bpe-10000-b46983c6f3/tinystories-train
TokenizedSource  tokenized/bpe-10000-b46983c6f3/tinystories-valid
MappedDataSet    mappeddatasets/mapped-2be543ce28
Pretraining      runs/runofex/pretraining/0-20000-2956000f6f


In [4]:
from local import declare_on_volume
report = declare_on_volume(pretraining, verbose=True, commit=True)  # commit=True publishes

sources/tinystories-train                         done  (drift: 73684a0 -> 5d6b791)
sources/tinystories-valid                         done  (drift: f9752b6 -> 5d6b791)
tokenizers/bpe-10000-b46983c6f3                   done  (drift: f9752b6 -> 5d6b791)
tokenized/bpe-10000-b46983c6f3/tinystories-train  done  (drift: f9752b6 -> 5d6b791)
tokenized/bpe-10000-b46983c6f3/tinystories-valid  done  (drift: f9752b6 -> 5d6b791)
mappeddatasets/mapped-2be543ce28                  done  (drift: f9752b6 -> 5d6b791)
runs/runofex/pretraining/0-20000-2956000f6f       declared

1 declared, 6 done
1 manifest written: runs/runofex/pretraining/0-20000-2956000f6f


## Benchmark

In [ ]:
result = run_profile(Profile(
    run_id=pretraining.run_id,
    resources=pretraining.allocated_resources,
    model=pretraining.model,
    optimizer=pretraining.training_parameters.optimizer,
    batch_size=pretraining.training_parameters.batch_size,
    seed=pretraining.training_parameters.seed,
    warmup_steps=5,
    profile_steps=4,
    autocast=pretraining.autocast,
    compile=pretraining.loop_config.compile,
    kernel_events=True,
    module_scopes=True,
    peak_memory=True,
    trace_memory=True,
))

In [ ]:
result.display_steps(steps=[3])

In [ ]:
step = 3
rows = result.events(step)
groups, kernels, unattributed_kernels, flop_sources = result.aggregate_kernels(step, depth=3)
display(groups[["kernel_count", "gpu_time_us", "gpu_span_us", "total_est_flops"]])

In [19]:
display(kernels[["name", "phase", "module_fqn", "owner_fqn", "gpu_start_us", "duration_us"]])

,name,phase,module_fqn,owner_fqn,gpu_start_us,duration_us
index,,,,,,
14,"void at::native::vectorized_gather_kernel<16, ...",forward,model.embedding,model.embedding,2966.336,13.120
2039,volta_sgemm_128x128_tn,forward,model.linear,model.linear,47889.222,2120.464
1987,void at::native::vectorized_elementwise_kernel...,forward,model.norm,model.norm,46852.541,6.688
1990,"void at::native::reduce_kernel<512, 1, at::nat...",forward,model.norm,model.norm,46965.659,14.944
1993,void at::native::vectorized_elementwise_kernel...,forward,model.norm,model.norm,47069.240,3.168
...,...,...,...,...,...,...
8755,void at::native::(anonymous namespace)::multi_...,optimizer_step,<unattributed>,NaN,150143.383,162.141
8915,void at::native::(anonymous namespace)::multi_...,optimizer_step,<unattributed>,NaN,150978.788,115.230
8919,void at::native::(anonymous namespace)::multi_...,optimizer_step,<unattributed>,NaN,151095.714,126.173


In [ ]:
kernel_id = kernels.index[0]
display(launch_chain(rows, kernel_id, format=True))